In [0]:
# Create all folders

from pyspark.dbutils import DBUtils
dbutils = DBUtils(spark)

base_path = "dbfs:/FileStore/databricks_project"

# folders = [
#     f"{base_path}/notebooks",
#     f"{base_path}/src/utils",
#     f"{base_path}/src/transformations",
#     f"{base_path}/tests"
# ]

# for folder in folders:
#     dbutils.fs.mkdirs(folder)
#     print(f"Folder created: {folder}")
# dbutils.fs.mkdirs(f"{base_path}/rules")
# dbutils.fs.mkdirs(f"{base_path}/reports")
# dbutils.fs.mkdirs(f"{base_path}/config")


In [0]:
%pip install -q pytest pytest-cov pytest-html pytest-xdist pytest-rerunfailures

Python interpreter will be restarted.
Python interpreter will be restarted.


In [0]:
# Write main_notebook.py inside notebooks/

notebook_content = '''# Databricks notebook source
print("Main Notebook Running")'''

dbutils.fs.put(f"{base_path}/notebooks/main_notebook.py", notebook_content, overwrite=True)
print("main_notebook.py written successfully")


Wrote 64 bytes.
✅ main_notebook.py written successfully


In [0]:
# Write spark_session.py inside src/utils/

spark_session_content = '''import os
from pyspark.sql import SparkSession

def get_spark(app_name: str = "PytestSession"):
    """
    Returns a Spark session. 
    If running inside pytest (subprocess), it sets a local master.
    """
    builder = SparkSession.builder.appName(app_name)

    # If running in a subprocess (like pytest), attach a local master
    if "PYTEST_CURRENT_TEST" in os.environ:
        builder = builder.master("local[*]")

    return builder.getOrCreate()

'''

dbutils.fs.put(f"{base_path}/src/utils/spark_session.py", spark_session_content, overwrite=True)
print("spark_session.py written successfully")


Wrote 457 bytes.
✅ spark_session.py written successfully


In [0]:
# Write transform_orders.py inside src/transformations/

transform_orders_content = '''from pyspark.sql.functions import col

def calculate_total_cost(df):
    """
    Adds a total_cost column by multiplying UnitPrice * Quantity.
    """
    return df.withColumn("total_cost", col("UnitPrice") * col("Quantity"))
'''

dbutils.fs.put(f"{base_path}/src/transformations/transform_orders.py", transform_orders_content, overwrite=True)
print("transform_orders.py written successfully")


Wrote 226 bytes.
✅ transform_orders.py written successfully


In [0]:
from pyspark.dbutils import DBUtils
dbutils = DBUtils(spark)

base_path = "dbfs:/FileStore/databricks_project"  # change if you use a different root
dbutils.fs.mkdirs(f"{base_path}/src/ingestion")

ingestion_content = '''\
from typing import Dict, Optional
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType
from pyspark.sql import functions as F

def ingest_csv(
    spark: SparkSession,
    input_path: str,
    output_path: str,
    schema: StructType,
    partition_col: str = "ingest_date",
    mode: str = "overwrite",
    drop_null_cols: Optional[list] = None,
    dedup_keys: Optional[list] = None,
) -> Dict[str, int]:
    """
    Small CSV ingestion:
    - Applies provided schema
    - Adds partition column if missing
    - Drops rows with nulls in drop_null_cols
    - De-duplicates on dedup_keys
    - Writes Parquet partitioned by partition_col
    - Returns simple stats
    """
    drop_null_cols = drop_null_cols or []
    dedup_keys = dedup_keys or []

    # Read permissively to keep tiny tests simple; capture corrupt rows
    df: DataFrame = (
        spark.read.format("csv")
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(schema)
        .load(input_path)
    )

    rows_in = df.count()
    corrupt_count = df.filter(F.col("_corrupt_record").isNotNull()).count() if "_corrupt_record" in df.columns else 0

    # Add partition col if absent
    if partition_col not in df.columns:
        df = df.withColumn(partition_col, F.current_date().cast("string"))

    # Filter nulls on required columns
    for c in drop_null_cols:
        df = df.filter(F.col(c).isNotNull())

    # Deduplicate
    if dedup_keys:
        before_dedupe = df.count()
        df = df.dropDuplicates(dedup_keys)
        deduped = before_dedupe - df.count()
    else:
        deduped = 0

    rows_out = df.count()

    # Write partitioned Parquet
    (
        df.write.mode(mode)
        .format("parquet")
        .partitionBy(partition_col)
        .save(output_path)
    )

    return {
        "rows_in": rows_in,
        "rows_out": rows_out,
        "deduped": deduped,
        "null_dropped": rows_in - corrupt_count - rows_out if rows_in >= rows_out else 0,
        "corrupt": corrupt_count,
    }
'''

dbutils.fs.put(f"{base_path}/src/ingestion/ingest.py", ingestion_content, overwrite=True)
print("ingest.py written successfully")


Wrote 2123 bytes.
ingest.py written successfully


In [0]:
# Write conftest.py inside tests/

conftest_content = '''import json
import os
from pathlib import Path
import pytest
from pyspark.sql import SparkSession

DEFAULT_CONFIG_PATH = Path("config/run_conditions.json")

def _load_cfg():
    cfg_path = Path(os.environ.get("PYTEST_CONFIG_FILE", DEFAULT_CONFIG_PATH))
    with open(cfg_path, "r", encoding="utf-8") as f:
        return json.load(f)

@pytest.fixture(scope="session")
def run_cfg():
    return _load_cfg()

@pytest.fixture(scope="session", autouse=True)
def _ensure_reports():
    Path("reports").mkdir(parents=True, exist_ok=True)

def pytest_configure(config):
    config.addinivalue_line("markers", "integration: integration tests")

def pytest_collection_modifyitems(config, items):
    cfg = _load_cfg()
    if not cfg.get("run_integration", True):
        skip_integ = pytest.mark.skip(reason="run_integration=false in config")
        for item in items:
            if "integration" in item.keywords:
                item.add_marker(skip_integ)

@pytest.fixture(scope="session")
def spark():
    spark = (
        SparkSession.builder
        .master("local[2]")
        .appName("small_ingestion_tests")
        .getOrCreate()
    )
    # keep logs quiet for tests
    spark.sparkContext.setLogLevel("ERROR")
    yield spark
    spark.stop()

'''

dbutils.fs.put(f"{base_path}/tests/conftest.py", conftest_content, overwrite=True)
print("conftest.py written successfully")


Wrote 1251 bytes.
conftest.py written successfully


In [0]:
# Write test_ingestion_unit.py inside tests/

test_ingestion = '''from pathlib import Path
import pytest
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from src.ingestion.ingest import ingest_csv

def _tiny_schema():
    return StructType([
        StructField("id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True),
        StructField("_corrupt_record", StringType(), True),  # to capture bad rows
    ])

def test_ingest_filters_nulls_and_dedup(spark, tmp_path):
    schema = _tiny_schema()
    input_dir = tmp_path / "in"
    output_dir = tmp_path / "out"
    input_dir.mkdir()

    # small CSV with nulls, duplicates
    (input_dir / "tiny.csv").write_text(
        "id,name,age\\n"
        "1,Alice,30\\n"
        "2,Bob,31\\n"
        ",NoID,40\\n"       # null id
        "2,Bob,31\\n",      # duplicate
        encoding="utf-8"
    )

    stats = ingest_csv(
        spark,
        str(input_dir),
        str(output_dir),
        schema=schema,
        partition_col="ingest_date",
        mode="overwrite",
        drop_null_cols=["id"],
        dedup_keys=["id"]
    )

    assert stats["rows_in"] == 4
    assert stats["deduped"] == 1
    assert stats["rows_out"] == 2  # 4 in - 1 null - 1 dup = 2
    assert (output_dir.exists() and any(output_dir.iterdir()))

def test_partition_column_created_when_missing(spark, tmp_path):
    schema = _tiny_schema()
    input_dir = tmp_path / "in"
    output_dir = tmp_path / "out"
    input_dir.mkdir()
    (input_dir / "tiny.csv").write_text("id,name,age\\n1,Alice,30\\n", encoding="utf-8")

    _ = ingest_csv(
        spark, str(input_dir), str(output_dir),
        schema=schema, partition_col="ingest_date",
        drop_null_cols=["id"], dedup_keys=["id"]
    )

    # Verify partitioned write
    parts = [p.name for p in output_dir.glob("ingest_date=*")]
    assert parts, "Expected partitioned folders like ingest_date=YYYY-MM-DD"

def test_corrupt_rows_counted(spark, tmp_path):
    schema = _tiny_schema()
    input_dir = tmp_path / "in"
    output_dir = tmp_path / "out"
    input_dir.mkdir()
    # malformed line (too many columns) will go to _corrupt_record
    (input_dir / "tiny.csv").write_text(
        "id,name,age\\n"
        "1,Alice,30\\n"
        "BAD,BAD,BAD,EXTRA\\n",
        encoding="utf-8"
    )

    stats = ingest_csv(
        spark, str(input_dir), str(output_dir),
        schema=schema, drop_null_cols=["id"], dedup_keys=["id"]
    )
    assert stats["corrupt"] == 1
'''


dbutils.fs.put(f"{base_path}/tests/test_ingestion_unit.py", test_ingestion, overwrite=True)
print("test_ingestion_unit.py written successfully")

tests/test_ingestion_integration.py


Wrote 2479 bytes.
test_ingestion_unit.py written successfully


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6693687986931550>:89
     86 dbutils.fs.put(f"{base_path}/tests/test_ingestion_unit.py", test_ingestion, overwrite=True)
     87 print("test_ingestion_unit.py written successfully")
---> 89 tests/test_ingestion_integration.py

NameError: name 'tests' is not defined

In [0]:
# test/test_ingestion _integration

test_ingestion_integration = '''import shutil
from pathlib import Path
import pytest
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F
from src.ingestion.ingest import ingest_csv

pytestmark = pytest.mark.integration

def _schema():
    return StructType([
        StructField("id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True),
        StructField("_corrupt_record", StringType(), True),
    ])

def test_end_to_end_partitioned_parquet(spark, tmp_path, run_cfg):
    assert "tiny" in run_cfg.get("datasets", []), "This test expects 'tiny' dataset enabled"

    input_dir = tmp_path / "tiny_in"
    output_dir = tmp_path / "tiny_out"
    input_dir.mkdir()

    (input_dir / "data.csv").write_text(
        "id,name,age\\n"
        "1,Alice,30\\n"
        "2,Bob,31\\n"
        "3,Charlie,29\\n",
        encoding="utf-8"
    )

    stats = ingest_csv(
        spark, str(input_dir), str(output_dir),
        schema=_schema(), partition_col="ingest_date",
        drop_null_cols=["id"], dedup_keys=["id"], mode="overwrite"
    )
    assert stats["rows_out"] == 3

    # Read back and validate schema + partition col exists
    df = spark.read.parquet(str(output_dir))
    assert "ingest_date" in df.columns
    assert set(["id","name","age"]).issubset(df.columns)

def test_idempotent_overwrite(spark, tmp_path):
    schema = _schema()
    input_dir = tmp_path / "in"
    output_dir = tmp_path / "out"
    input_dir.mkdir()
    (input_dir / "data.csv").write_text("id,name,age\\n1,Alice,30\\n", encoding="utf-8")

    # first run
    s1 = ingest_csv(spark, str(input_dir), str(output_dir), schema, dedup_keys=["id"], drop_null_cols=["id"], mode="overwrite")
    # second run with same input and overwrite mode should keep count stable
    s2 = ingest_csv(spark, str(input_dir), str(output_dir), schema, dedup_keys=["id"], drop_null_cols=["id"], mode="overwrite")
    assert s1["rows_out"] == s2["rows_out"] == 1
'''


dbutils.fs.put(f"{base_path}/tests/test_ingestion_integration.py", test_ingestion_integration, overwrite=True)
print("test_ingestion_integration.py written successfully")



Wrote 2014 bytes.
test_ingestion_integration.py written successfully


In [0]:
# Write requirements.txt at project root

requirements_content = '''pyspark>=3.4
pytest>=8.0
pytest-html>=4.1
'''

dbutils.fs.put(f"{base_path}/requirements.txt", requirements_content, overwrite=True)
print("requirements.txt written successfully")


Wrote 42 bytes.
requirements.txt written successfully


In [0]:
# Write pytest.ini at project root

pytest_ini_content = '''[pytest]
testpaths = tests
addopts =
    -ra
    --junitxml=reports/junit.xml
    --html=reports/report.html
    --self-contained-html
python_files = test_*.py

'''

dbutils.fs.put(f"{base_path}/pytest.ini", pytest_ini_content, overwrite=True)
print("pytest.ini written successfully")


Wrote 161 bytes.
pytest.ini written successfully


In [0]:
# Write README.md at project root

readme_content = '''# Databricks Pytest Project 🧪

This is a production-ready pytest project structure for Databricks Community Edition.

## Structure
- notebooks/
- src/
- tests/
- requirements.txt
- pytest.ini

## Run Tests
```bash
%sh
cd /dbfs/FileStore/databricks_project
pytest -v
pytest -v
'''

dbutils.fs.put(f"{base_path}/README.md", readme_content, overwrite=True)
print("README.md written successfully")


Wrote 279 bytes.
✅ README.md written successfully


In [0]:
%fs ls /FileStore/databricks_project

path,name,size,modificationTime
dbfs:/FileStore/databricks_project/README.md,README.md,279,1761566877000
dbfs:/FileStore/databricks_project/notebooks/,notebooks/,0,0
dbfs:/FileStore/databricks_project/pytest.ini,pytest.ini,82,1761566786000
dbfs:/FileStore/databricks_project/requirements.txt,requirements.txt,7,1761566756000
dbfs:/FileStore/databricks_project/src/,src/,0,0
dbfs:/FileStore/databricks_project/tests/,tests/,0,0


In [0]:
dbutils.fs.ls("/FileStore/databricks_project/tests")


Out[4]: [FileInfo(path='dbfs:/FileStore/databricks_project/tests/conftest.py', name='conftest.py', size=1251, modificationTime=1761585698000),
 FileInfo(path='dbfs:/FileStore/databricks_project/tests/test_ingestion_integration.py', name='test_ingestion_integration.py', size=2008, modificationTime=1761586098000),
 FileInfo(path='dbfs:/FileStore/databricks_project/tests/test_ingestion_unit.py', name='test_ingestion_unit.py', size=2470, modificationTime=1761585878000),
 FileInfo(path='dbfs:/FileStore/databricks_project/tests/test_transform_orders.py', name='test_transform_orders.py', size=398, modificationTime=1761566686000)]

In [0]:
%sh
rm -rf /databricks/driver/tests


In [0]:
%sh
rm -rf /databricks/driver/src

In [0]:
# Copy all files from FileStore to /databricks/driver/tests
dbutils.fs.cp("dbfs:/FileStore/databricks_project/tests", "file:/databricks/driver/tests", recurse=True)


Out[26]: True

In [0]:
%sh
ls /databricks/driver/tests/


conftest.py
test_ingestion_integration.py
test_ingestion_unit.py
test_transform_orders.py


In [0]:
dbutils.fs.cp(
    "dbfs:/FileStore/databricks_project/src",
    "file:/databricks/driver/src",
    recurse=True
)

Out[27]: True

In [0]:
import pytest

# Set up the Python path so 'src' is found
import sys
sys.path.append("/databricks/driver")

# Run pytest inside this Python process
pytest.main(["/databricks/driver/tests", "-v", "-s"])
